# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PaNavar369/Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The ranked action queue is designed to help prioritise pages for human review rather than automatically change content. Pages are ranked using observed search signals from the available dataset. The highest-priority pages are those showing combinations of aging content, declining trends, high visibility opportunity, or weak CTR.

Each recommendation includes a reason code so that a content reviewer can understand why the page was prioritised. The reason codes are directional indicators based on observed signals and should not be interpreted as causal explanations.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/PaNavar369/Internship/main/data/raw/content_refresh_anonymized.csv"
)

# Create a working copy
playbook_df = df.copy()

# Fill missing values for the fields used in the action rules
playbook_df["days_since_last_update"] = playbook_df[
    "days_since_last_update"
].fillna(playbook_df["days_since_last_update"].median())

playbook_df["ctr"] = playbook_df["ctr"].fillna(0)

playbook_df["avg_position"] = playbook_df["avg_position"].fillna(
    playbook_df["avg_position"].median()
)

playbook_df["impressions_90d"] = playbook_df[
    "impressions_90d"
].fillna(0)


# Create reason codes
def assign_reason(row):

    reasons = []

    # Aging / refresh signal
    if row["days_since_last_update"] >= 365:
        reasons.append("STALE_CONTENT")

    # High visibility but weak CTR
    if row["impressions_90d"] >= 1000 and row["ctr"] < 0.05:
        reasons.append("HIGH_IMPRESSIONS_LOW_CTR")

    # Ranking opportunity
    if 4 <= row["avg_position"] <= 15:
        reasons.append("POSITION_OPPORTUNITY")

    # Declining trend
    if row["trend_direction"] == "down":
        reasons.append("DECLINING_TREND")

    if not reasons:
        reasons.append("MONITOR")

    return " | ".join(reasons)


playbook_df["reason_code"] = playbook_df.apply(
    assign_reason,
    axis=1
)


# Create an action recommendation
def assign_action(row):

    reasons = row["reason_code"]

    if "DECLINING_TREND" in reasons and "STALE_CONTENT" in reasons:
        return "Human review for content refresh"

    elif "HIGH_IMPRESSIONS_LOW_CTR" in reasons:
        return "Human review of title, metadata and search intent"

    elif "POSITION_OPPORTUNITY" in reasons:
        return "Review optimisation opportunity"

    elif "STALE_CONTENT" in reasons:
        return "Review freshness and content accuracy"

    else:
        return "Monitor - no immediate action"


playbook_df["recommended_action"] = playbook_df.apply(
    assign_action,
    axis=1
)


# Priority score for ranking the queue
playbook_df["priority_score"] = 0

playbook_df.loc[
    playbook_df["trend_direction"] == "down",
    "priority_score"
] += 3

playbook_df.loc[
    playbook_df["days_since_last_update"] >= 365,
    "priority_score"
] += 2

playbook_df.loc[
    (playbook_df["impressions_90d"] >= 1000) &
    (playbook_df["ctr"] < 0.05),
    "priority_score"
] += 3

playbook_df.loc[
    (playbook_df["avg_position"] >= 4) &
    (playbook_df["avg_position"] <= 15),
    "priority_score"
] += 2


# Rank pages
ranked_queue = playbook_df.sort_values(
    by="priority_score",
    ascending=False
).copy()

ranked_queue["rank"] = range(1, len(ranked_queue) + 1)


# Display the top 20 recommendations
queue_columns = [
    "rank",
    "content_id",
    "recommended_action",
    "reason_code",
    "priority_score",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d",
    "trend_direction"
]

display(
    ranked_queue[queue_columns].head(20)
)

,rank,content_id,recommended_action,reason_code,priority_score,days_since_last_update,ctr,avg_position,impressions_90d,trend_direction
8180,1,content_bc2f4f6ac33c,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,104,0.00,6.8,1191,down
24351,2,content_7c98dadae273,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,20,0.03,12.2,3310,down
20096,3,content_d9fd8bb80909,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,104,0.00,13.9,1371,down
8150,4,content_bb039f652a92,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,104,0.04,11.8,4681,down
8143,5,content_b929edda93a0,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,22,0.00,9.5,2053,down
26700,6,content_8d387d58f71f,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,8,0.00,5.7,1032,down
29361,7,content_3a42e62946d6,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,104,0.00,9.8,1981,down
17434,8,content_0a08a44b7770,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,22,0.03,4.6,3233,down
8077,9,content_a0404909ecca,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,104,0.00,14.1,1115,down
11346,10,content_2dfe0a3cc206,"Human review of title, metadata and search intent",HIGH_IMPRESSIONS_LOW_CTR | POSITION_OPPORTUNIT...,8,22,0.02,8.8,16690,down


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This content action playbook is intended for content teams or SEO reviewers who need a structured way to prioritise pages for review. The ranked queue combines observed freshness, CTR, average position, impressions, and trend signals to identify pages that may deserve attention first.

The playbook is a decision-support tool. It does not automatically publish, rewrite, delete, or modify content. The model and rules do not prove that any individual signal causes ranking changes.

Validation results also limit how strongly the recommendations can be interpreted. The Decision Tree achieved 60.15% accuracy under the Week-5 random split and 52.21% accuracy under the client-grouped split. The lower grouped result suggests that performance estimates depend on validation design and that recommendations should be reviewed carefully before action.

The playbook should not be treated as a system for predicting Google rankings or as evidence of causal relationships. It is intended to help humans prioritise investigation using observed and measured signals.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Every recommendation requires human review before action. A reviewer should check the page's business importance, search intent, content accuracy, recent changes, seasonality, and whether the available data is complete.

A high priority score does not mean that a page should automatically be rewritten. The score only indicates that the page matches one or more observed prioritisation signals.

In [2]:
human_review_rules = [
    "Check that the page is still strategically important.",
    "Check whether the page matches current search intent.",
    "Check for recent content or technical changes.",
    "Check whether the page has seasonal performance patterns.",
    "Check data completeness before acting on the recommendation.",
    "Review the reason codes rather than relying only on priority rank.",
    "Confirm that the proposed action is appropriate for the page."
]

for rule in human_review_rules:
    print("-", rule)

- Check that the page is still strategically important.
- Check whether the page matches current search intent.
- Check for recent content or technical changes.
- Check whether the page has seasonal performance patterns.
- Check data completeness before acting on the recommendation.
- Review the reason codes rather than relying only on priority rank.
- Confirm that the proposed action is appropriate for the page.


In [3]:
no_go_list = [
    "Do not automatically publish new content.",
    "Do not automatically rewrite existing content.",
    "Do not automatically delete pages.",
    "Do not automatically change titles or metadata.",
    "Do not make causal claims from the recommendation score.",
    "Do not claim to predict future Google rankings.",
    "Do not act automatically when data is missing or incomplete.",
    "Do not use the queue as the only basis for high-cost editorial decisions."
]

print("NO-GO LIST:")
for item in no_go_list:
    print("-", item)

NO-GO LIST:
- Do not automatically publish new content.
- Do not automatically rewrite existing content.
- Do not automatically delete pages.
- Do not automatically change titles or metadata.
- Do not make causal claims from the recommendation score.
- Do not claim to predict future Google rankings.
- Do not act automatically when data is missing or incomplete.
- Do not use the queue as the only basis for high-cost editorial decisions.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


The recommendations may become stale if search behaviour, content patterns, or data collection changes. The playbook should therefore be monitored rather than treated as a permanent production system.

A review or retraining process should be triggered when model performance falls, important input distributions change, or the data available to the system changes significantly.

In [4]:
monitoring_triggers = {

    "validation_performance": (
        "Review the model if future validation performance "
        "falls materially below the current grouped benchmark of 52.21%."
    ),

    "feature_drift": (
        "Review the model if the distributions of CTR, average position, "
        "impressions, or days since update change substantially."
    ),

    "data_quality": (
        "Review recommendations if missing values or incomplete search "
        "data increase materially."
    ),

    "content_pattern_change": (
        "Review the playbook if new content types or search patterns "
        "are not represented in the original training data."
    ),

    "recommendation_outcomes": (
        "Review whether high-priority recommendations consistently "
        "produce useful editorial investigations."
    )
}

for trigger, explanation in monitoring_triggers.items():
    print(f"\n{trigger.upper()}")
    print(explanation)


VALIDATION_PERFORMANCE
Review the model if future validation performance falls materially below the current grouped benchmark of 52.21%.

FEATURE_DRIFT
Review the model if the distributions of CTR, average position, impressions, or days since update change substantially.

DATA_QUALITY
Review recommendations if missing values or incomplete search data increase materially.

CONTENT_PATTERN_CHANGE
Review the playbook if new content types or search patterns are not represented in the original training data.

RECOMMENDATION_OUTCOMES
Review whether high-priority recommendations consistently produce useful editorial investigations.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



The queue should also be interpreted using cost and expected value. A high-priority page is not automatically worth updating if the editorial cost is high and the potential business value is low.

Before acting, reviewers should consider:

Expected visibility opportunity
Business importance of the page
Editorial effort required
Risk of making unnecessary changes
Whether the recommendation is supported by multiple signals

A lower-ranked page with high business value may deserve attention before a higher-ranked page with limited value. The ranking queue supports prioritisation but does not replace editorial judgement.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.